In [1]:
import dspy
from typing import List, Optional, Literal
from pydantic import BaseModel,Field

In [69]:
from openai import OpenAI
import os
import dspy

lm = dspy.LM(
    model="openai/gpt-5.2",
    api_key=os.environ["OPENAI_API_KEY"],
)
dspy.configure(lm=lm)

### POLICY EXTRACTION ###

In [67]:
class ExtractedPolicy(BaseModel):
    # Core extraction
    policy_statement: str = Field(
        "A concise, self-contained summary of the policy commitment. "
        "A policy statement must describe a deliberate action, resource allocation, regulatory change, or quantifiable target. "
        "Rewrite the text into a clear, active-voice statement")
    verbatim_text: str = Field("Details of the policy must be quoted verbatim from the source (no paraphrasing), try to include 2-3 sentences. but include the full details of the policy")
    # Hiearchy field thats new, could turn into an enum to be more strict
    section_header: str = Field(
        "The nearest section or subsection heading this policy falls under. "
        "Copy exactly as it appears in the document. If multiple levels exist, "
        "provide them separated by ' > ' (e.g., 'Energy Sector > Renewable Energy Targets'). "
        "If no header is identifiable, return 'No header found'."
    )
    preceding_paragraph: str = Field(
        "The paragraph immediately before this policy in the source document. "
        "Quote verbatim. This often contains the framing, initiative name, or "
        "rationale that connects related policies together. "
        "If the policy is the first item in a section, return the section introduction."
    )
    policy_group_name: Optional[str] = Field(
        default=None,
        description=(
            "If this policy is explicitly listed as part of a named plan, strategy, "
            "programme, or initiative in the document, provide that name exactly as written. "
            "Examples: 'National E-Mobility Programme', 'Green Building Action Plan'. "
            "None if the policy is not part of any named group."
        )
    )
    sector: str = Field(
        "The climate/policy sector this falls under. Use one of: "
        "'energy', 'transport', 'buildings', 'industry', 'agriculture', "
        "'forestry_land_use', 'waste', 'water', 'cross_cutting', 'adaptation', 'finance'."
    )

    # Soundness indicators
    has_quantifiable_target: str = Field("Indicate 'Yes' or 'No'. If 'Yes', provide details on the target (e.g., '40% reduction by 2030').")
    has_timeline: str = Field("Indicate 'Yes' or 'No'. If 'Yes', provide details on the timeline (e.g., 'by 2030').")
    has_binding_mechanism: str = Field("Indicate 'Yes' or 'No'. If 'Yes', provide details on the binding mechanism (e.g., 'legal mandate').")
    has_spatial_specificity: str = Field("Indicate 'Yes' or 'No'. If 'Yes', provide details on the spatial specificity (e.g., 'national', 'state-level').")

    # Substance flag NEW
    has_implementation_pathway: str = Field(
        "Indicate 'Yes' or 'No'. 'Yes' if the policy specifies HOW it will be achieved "
        "(named programs, responsible entities, funding, intermediate steps). "
        "A numeric target alone without a mechanism is 'No'."
    )

    extraction_rationale: str = Field("Why this qualifies as a policy, based on the criteria provided in the initial prompt, any flags I should watch out for")

In [8]:
class DocumentMetadata(BaseModel):
    country: str 
    state_or_province: Optional[str]
    city: Optional[str] 


In [64]:
class PolicyExtractionSignature(dspy.Signature):
    """
    Extract climate policies from document text.
    
    DEFINITION OF A POLICY
    A policy is a STATED COMMITMENT by a governing body to achieve a defined 
    outcome through deliberate action, resource allocation, or regulatory change.
    
    A policy is NOT:
    - Background information or problem descriptions
    - Statements of current conditions
    - Aspirations without any specified action
    - Descriptions of what other actors might do
    
    WHAT MAKES SOMETHING EXTRACTABLE
    Extract a statement as a policy if it contains AT LEAST ONE of:
    
    1. QUANTIFIABLE TARGET: Numbers with units and/or deadlines
        -"Reduce emissions 40% by 2030"
       -"Install 5GW of solar capacity"
       -"Retrofit 10,000 buildings"
    
    2. BINDING MECHANISM: Legal or regulatory force
       -"Mandatory building codes requiring..."
       -"Ban on single-use plastics"
       -"Carbon tax of $25/ton"
    
    3. SPECIFIC INTERVENTION: Named program, technology, or action
       -"National E-Mobility Program"
       -"Phase-out of coal-fired power plants"
       -"Expansion of bus rapid transit network"
    
    4. RESOURCE ALLOCATION: Committed funding or investment
       -"$500M allocated for renewable energy"
       -"Dedicated climate adaptation fund"
    
    EDGE CASES: EXTRACT BUT FLAG
    - Vague commitments ("promote", "encourage", "explore") — extract if 
      they reference a specific sector or mechanism, note vagueness
    - Strategies/roadmaps without targets — extract if they name concrete 
      action areas
    - Conditional commitments ("subject to international funding") — extract,
      note conditionality
    
    DO NOT EXTRACT
    - Pure context: "Climate change threatens coastal areas"
    - Current state: "The country currently has 2GW of wind capacity"  
    - Process statements: "Stakeholder consultations will be held"
    - Vague aspirations with no anchor: "We aspire to a green future"
    
    HIERARCHY RECOGNITION
    Many documents contain initiatives or action plans with sub-policies.
    
    - If a section header or paragraph introduces a broad plan (e.g., 
      "National E-Mobility Program") followed by specific measures, 
      the broad plan is an 'initiative' and each measure is an 'action'.
    - An 'action' should reference its parent in parent_initiative using
      the exact name from the document.
    - A policy with no parent above it and no children below it is 'standalone'.
    
    Example:
      "Transport Decarbonization Strategy" -> policy_type: initiative
        "Electrify 50% of bus fleet by 2028" -> policy_type: action, 
         parent_initiative: "Transport Decarbonization Strategy"

    If no policies are found in the chunk, return an empty list.
    """
    document_text: str = dspy.InputField(
        desc="Text extracted from a climate policy document (NDC, action plan, etc.)"
    )
    document_metadata: DocumentMetadata = dspy.InputField(
        desc="Document name, country, year, and any known section context"
    )
    
    policies: List[ExtractedPolicy] = dspy.OutputField(
        desc="List of extracted policies with metadata; empty list if none found"
    )


In [57]:
class PolicyExtractor(dspy.Module):
    """
    Extracts structured policy objects from document text.
    
    Designed to run on chunks/sections of a larger document.
    Handles the full extraction task: identification, extraction, 
    and preliminary soundness assessment.
    """
    
    def __init__(self):
        self.extract = dspy.ChainOfThought(PolicyExtractionSignature)
    
    def forward(self, document_text: str, document_metadata: str = "") -> List[ExtractedPolicy]:
        result = self.extract(
            document_text=document_text,
            document_metadata=document_metadata
        )
        return result.policies

In [11]:
from docling.document_converter import DocumentConverter
converter= DocumentConverter()

In [13]:
document = converter.convert("../pdfs/Chicago-CAP-071822.pdf")

2026-02-14 16:49:39,671 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-02-14 16:49:40,084 - INFO - Going to convert document batch...
2026-02-14 16:49:40,085 - INFO - Processing document Chicago-CAP-071822.pdf
2026-02-14 16:52:19,828 - INFO - Finished converting document Chicago-CAP-071822.pdf in 160.98 sec.


In [14]:
markdown_version = document.document.export_to_markdown()

In [15]:
d_metadata = DocumentMetadata(
    country="USA",
    state_or_province="Illinois",
    city="Chicago"
)


In [70]:
policy_extractor = PolicyExtractor()
extracted_policies = policy_extractor(
    document_text=markdown_version,
    document_metadata=d_metadata
)


In [71]:
extracted_policies


[ExtractedPolicy(policy_statement='Reduce Chicago’s greenhouse gas emissions by at least 62% by 2040 from 2017 levels.', verbatim_text="“The 2022 CAP aims to chart an equitable path to reduce Chicago's GHG emissions by a minimum of 62% by 2040. ... this pathway prioritizes improving the lives of all Chicagoans ... by 2040.”", section_header='GHG Reduction Targets', preceding_paragraph="## GHG REDUCTION Chicago's\n\n## TARGETS", policy_group_name='2022 Climate Action Plan (CAP)', sector='cross_cutting', has_quantifiable_target="Yes: reduce Chicago's GHG emissions by a minimum of 62% by 2040 (from 2017 inventory levels).", has_timeline='Yes: by 2040.', has_binding_mechanism='No (stated as a plan aim/target; no ordinance/mandate specified here).', has_spatial_specificity='Yes: City of Chicago (citywide).', has_implementation_pathway='Yes (the CAP specifies pillars/strategies/actions and an implementation/accountability framework).', extraction_rationale='Quantifiable citywide GHG reductio

In [ ]:
# this version actually ends up over counting lots of policies

In [72]:
import json

with open("extracted_policies_chicago_intermediate_5_2_final", "w") as f:
    json.dump([p.dict() for p in extracted_policies], f, indent=2)
    

/var/folders/f5/bb79s3gs38q_wp4_2kvwbz9w0000gn/T/ipykernel_31367/982157091.py:4: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  json.dump([p.dict() for p in extracted_policies], f, indent=2)


In [ ]:
class PolicyCluster(BaseModel):
    cluster_name: str = Field(
        description="The name of the action plan or initiative that groups these policies together. "
        "Use the exact name as it appears in the document (from policy_group_name or section context). "
        "Use 'Standalone Policy' for policies not part of any broader initiative."
    )
    cluster_type: Literal["action_plan", "standalone"] = Field(
        description="'action_plan' if multiple policies explicitly share a named initiative. "
        "'standalone' if this is an individual policy not part of a broader plan."
    )
    policy_ids: List[int] = Field(
        description="Indices of policies that belong to this cluster (0-indexed from input list)"
    )
    rationale: str = Field(
        description="Brief explanation of why these policies were grouped together "
        "(e.g., 'All explicitly listed under policy_group_name: National E-Mobility Programme'). "
        "For standalone policies, state 'No shared action plan identified.'"
    )

class PolicyClusteringSignature(dspy.Signature):
    """
    Identify policies that belong to explicitly named action plans or initiatives.
    
    CONSERVATIVE CLUSTERING APPROACH
    Default assumption: Most policies are STANDALONE unless there is clear evidence 
    they belong to a specific, named action plan.
    
    WHEN TO CLUSTER (All conditions must be met):
    
    1. EXPLICIT NAMED GROUPING (Required)
       Multiple policies must share the EXACT SAME policy_group_name that is:
       - Not null
       - A specific initiative/program name (e.g., "National E-Mobility Programme")
       - NOT a generic category (e.g., don't cluster just because sector="energy")
    
    2. MINIMUM GROUP SIZE
       At least 2 policies must share the same policy_group_name to form a cluster.
       A single policy with a policy_group_name is still STANDALONE.
    
    ALTERNATIVE CLUSTERING CRITERIA (Use only if very clear):
    - Multiple policies under a highly specific section_header (4+ levels deep) 
      with a named program in the header itself
      Example: "Climate Action > Transport > Electric Vehicle Rollout Programme > Phase 1"
    
    - Multiple policies that quote the IDENTICAL preceding_paragraph which explicitly 
      introduces a named initiative and lists the following policies as part of it
    
    DEFAULT TO STANDALONE if:
    - policy_group_name is null or None
    - Only sharing a generic section_header (1-3 levels)
    - Precedence paragraph is generic context, not an initiative introduction
    - Only one policy has a given policy_group_name
    - Any uncertainty about whether they belong together
    
    CLUSTERING STRICTNESS
    Be conservative. When in doubt, mark as standalone.
    Better to under-cluster than over-cluster.
    
    OUTPUT FORMAT
    Return one PolicyCluster per group. For standalone policies, each gets its own 
    cluster with cluster_type="standalone" and cluster_name="Standalone Policy".
    """
    
    extracted_policies: List[ExtractedPolicy] = dspy.InputField(
        desc="List of extracted policies with section_header, preceding_paragraph, and policy_group_name fields"
    )
    
    clusters: List[PolicyCluster] = dspy.OutputField(
        desc="List of policy clusters. Most should be standalone; only group if explicitly part of same named initiative"
    )

class PolicyClusterer(dspy.Module):
    """
    Conservatively groups policies into action plans.
    Only clusters when there's explicit evidence of a shared initiative.
    Most policies remain standalone.
    """
    
    def __init__(self):
        self.cluster = dspy.ChainOfThought(PolicyClusteringSignature)
    
    def forward(self, extracted_policies: List[ExtractedPolicy]) -> List[PolicyCluster]:
        result = self.cluster(extracted_policies=extracted_policies)
        return result.clusters

def add_cluster_to_policies(
    policies: List[ExtractedPolicy], 
    clusters: List[PolicyCluster]
) -> List[dict]:
    """
    Adds cluster_name and cluster_type to each policy.
    """
    enriched_policies = []
    
    for idx, policy in enumerate(policies):
        policy_dict = policy.dict()
        
        # Find which cluster this policy belongs to
        for cluster in clusters:
            if idx in cluster.policy_ids:
                policy_dict['cluster_name'] = cluster.cluster_name
                policy_dict['cluster_type'] = cluster.cluster_type
                break
        else:
            # Safety fallback
            policy_dict['cluster_name'] = 'Standalone Policy'
            policy_dict['cluster_type'] = 'standalone'
        
        enriched_policies.append(policy_dict)
    
    return enriched_policies